In [27]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("day-25")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)

In [28]:
customers_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/customers.csv')
orders_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/orders.csv')
products_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/products.csv')

**Problem 1** | Easy

Standardize customer names

Create a full_name column combining first_name and last_name in Title Case, trimmed of extra whitespace, separated by a single space.

In [29]:
customers_df.withColumn(
    'full_name',
    F.initcap(
        F.concat_ws(
            ' ',
            F.trim('first_name'),
            F.trim('last_name')
        )
    )
).select('full_name').show(5, truncate=False)

+--------------+
|full_name     |
+--------------+
|James Anderson|
|Maria Garcia  |
|Robert Johnson|
|Linda Martinez|
|Michael Brown |
+--------------+
only showing top 5 rows


**Problem 2** | Easy

Extract email domain and flag corporate emails

Extract the domain from each customer's email. Flag as is_gmail = true if the domain contains "gmail", false otherwise.

In [30]:
customers_df.select(
    "customer_id", "email",
    F.split(F.col("email"), "@")[1].alias("domain"),
    F.split(F.col("email"), "@")[1].contains("gmail").alias("is_gmail")
).show(5, truncate=False)

+-----------+------------------------+---------+--------+
|customer_id|email                   |domain   |is_gmail|
+-----------+------------------------+---------+--------+
|C001       |james.anderson@email.com|email.com|false   |
|C002       |maria.garcia@email.com  |email.com|false   |
|C003       |robert.johnson@email.com|email.com|false   |
|C004       |linda.martinez@email.com|email.com|false   |
|C005       |michael.brown@email.com |email.com|false   |
+-----------+------------------------+---------+--------+
only showing top 5 rows


**Problem 3** | Medium

Orders placed on weekends

Find all orders placed on a Saturday or Sunday. Return order_id, order_date, and a day_name column. How many weekend orders are there?

In [31]:
orders_df2 = orders_df.withColumn(
    'day_name',
    F.date_format('order_date', 'EEEE')
).filter(
    F.col('day_name').isin(['Saturday', 'Sunday'])
).select(
    'order_id',
    'order_date',
    'day_name'
)
orders_df2.show(5, truncate=False)
orders_df2.groupby('day_name').\
    agg(F.count("*").alias('total_weekend_orders')).\
    show()

+--------+----------+--------+
|order_id|order_date|day_name|
+--------+----------+--------+
|O0002   |2023-01-07|Saturday|
|O0005   |2023-01-15|Sunday  |
|O0008   |2023-01-22|Sunday  |
|O0010   |2023-01-28|Saturday|
|O0012   |2023-02-05|Sunday  |
+--------+----------+--------+
only showing top 5 rows


+--------+--------------------+
|day_name|total_weekend_orders|
+--------+--------------------+
|Saturday|                  14|
|  Sunday|                  14|
+--------+--------------------+



**Problem 4** | Medium

Customers who joined in their first 90 days as VIP

For each customer, find orders placed within 90 days of their signup_date. Label these as "Early Adopter Purchase", all others as "Regular Purchase".

In [32]:
joined_df=orders_df.join(customers_df,on='customer_id',how='inner')
joined_df.withColumn(
    "Label",
    F.when(
        (F.col('order_date') >= F.col('signup_date')) &
        (F.col('order_date') <= F.date_add('signup_date', 90)),
        "Early Adopter Purchase"
    ).otherwise("Regular")
).show(5,truncate=False)

+-----------+--------+----------+----------+--------+----------+------------+---------+--------------+-------+----------+---------+------------------------+-----------+-----+-------+-----------+----------+-------+
|customer_id|order_id|product_id|order_date|quantity|unit_price|discount_pct|status   |payment_method|region |first_name|last_name|email                   |city       |state|country|signup_date|segment   |Label  |
+-----------+--------+----------+----------+--------+----------+------------+---------+--------------+-------+----------+---------+------------------------+-----------+-----+-------+-----------+----------+-------+
|C001       |O0001   |P001      |2023-01-05|2       |1299.99   |10          |Delivered|Credit Card   |East   |James     |Anderson |james.anderson@email.com|New York   |NY   |USA    |2021-03-15 |Enterprise|Regular|
|C002       |O0002   |P005      |2023-01-07|1       |449.99    |0           |Delivered|PayPal        |West   |Maria     |Garcia   |maria.garcia@

**Problem 5** | Hard

Mask email addresses for a data export

Create a masked version of each customer's email that shows only the first 2 characters of the username, then asterisks, then the full domain — e.g. "ja****@email.com" for james.anderson@email.com. Use string functions only, no UDF.

In [33]:
customers_df.withColumn("username", F.split(F.col("email"), "@")[0]) \
    .withColumn("domain", F.split(F.col("email"), "@")[1]) \
    .withColumn(
        "masked_email",
        F.concat(F.substring(F.col("username"), 1, 2), F.lit("****@"), F.col("domain"))
    ).select("email", "masked_email") \
    .show(5, truncate=False)

+------------------------+----------------+
|email                   |masked_email    |
+------------------------+----------------+
|james.anderson@email.com|ja****@email.com|
|maria.garcia@email.com  |ma****@email.com|
|robert.johnson@email.com|ro****@email.com|
|linda.martinez@email.com|li****@email.com|
|michael.brown@email.com |mi****@email.com|
+------------------------+----------------+
only showing top 5 rows


**Problem 6** | Hard

Generate a fiscal quarter label

Assume the company's fiscal year starts in April (so April–June = Q1, July–Sept = Q2, Oct–Dec = Q3, Jan–March = Q4 of the previous fiscal year). For each order, generate a label like "FY2023-Q3" based on order_date.

In [34]:
orders_df.withColumn(
    "fiscal_quarter",
    F.when((F.month("order_date") >= 4) & (F.month("order_date") <= 6),
           F.concat(F.lit("FY"), F.year("order_date"), F.lit("-Q1")))
     .when((F.month("order_date") >= 7) & (F.month("order_date") <= 9),
           F.concat(F.lit("FY"), F.year("order_date"), F.lit("-Q2")))
     .when((F.month("order_date") >= 10) & (F.month("order_date") <= 12),
           F.concat(F.lit("FY"), F.year("order_date"), F.lit("-Q3")))
     .otherwise(F.concat(F.lit("FY"), F.year("order_date") - 1, F.lit("-Q4")))
).select("order_id", "order_date", "fiscal_quarter").orderBy("order_date").show(15)

+--------+----------+--------------+
|order_id|order_date|fiscal_quarter|
+--------+----------+--------------+
|   O0001|2023-01-05|     FY2022-Q4|
|   O0002|2023-01-07|     FY2022-Q4|
|   O0003|2023-01-10|     FY2022-Q4|
|   O0004|2023-01-12|     FY2022-Q4|
|   O0005|2023-01-15|     FY2022-Q4|
|   O0006|2023-01-18|     FY2022-Q4|
|   O0007|2023-01-20|     FY2022-Q4|
|   O0008|2023-01-22|     FY2022-Q4|
|   O0009|2023-01-25|     FY2022-Q4|
|   O0010|2023-01-28|     FY2022-Q4|
|   O0011|2023-02-02|     FY2022-Q4|
|   O0012|2023-02-05|     FY2022-Q4|
|   O0013|2023-02-08|     FY2022-Q4|
|   O0014|2023-02-10|     FY2022-Q4|
|   O0015|2023-02-14|     FY2022-Q4|
+--------+----------+--------------+
only showing top 15 rows


In [35]:
spark.stop()